# Prepare the Share Class Hedge Sheet

In [3]:
# Import libraries

%run utilities.ipynb # for the timediff() function
start_time          = time.time()
start_time_overlord = start_time
print('Importing libraries ...')

import pandas as pd
import numpy as np
import xlwings as xw
from pathlib import Path
import os, shutil, re
import win32com.client as win32 # library to convert xls to xlsx

print(f'Importing libraries completed: {timediff(start_time, time.time())}', '\n')

Importing libraries ...
Importing libraries completed: 0.0sec 



In [4]:
# Set location paths for the sheets to be used

start_time = time.time()
print('Setting up paths ...')

pth      = r'P:\Investment Operations\GRC\Compliance\PGF UCITS Share Class Hedges'
pth_dl   = str(Path.home() / 'Downloads')
pth_tmpl = r'P:\Working Folders\Hilton\py\pgf.xlsx' # hedge calculation template

print(f'Setting up paths completed: {timediff(start_time, time.time())}', '\n')

Setting up paths ...
Setting up paths completed: 0.0sec 



In [12]:
# Get report date and selected summary sheet option

start_time = time.time()
print('Getting the reporting date and the compartive prior reporting date ...')

# extract override report date from cell "E1" in dervs sheet of py_reports.xlsm
df      = pd.read_excel(r'P:\Investment Operations\GRC\Compliance\Daily\py_reports.xlsm', sheet_name = 'hdgs', header = None, usecols = 'D', nrows = 1)
k       = df.iloc[0,0]
rDate   = prior_working_day() if not isinstance(k, datetime.datetime) else k  # prior working day or report date override; of type datetime.datetime()
rptDate = rDate.strftime("%d%b%Y")                          # get report date in datetime ddMmmYYYY format for file names
prrDate = prior_working_day(rDate)

print(f' Reporting date is {rDate.strftime("%a %d %b %Y")}, with comparative date {prrDate.strftime("%a %d %b %Y")}')
print(f'Getting the reporting date and the comparative prior reporting date completed: {timediff(start_time, time.time())}')

Getting the reporting date and the compartive prior reporting date ...
 Reporting date is Wed 08 Jan 2025, with comparative date Tue 07 Jan 2025
Getting the reporting date and the compartive prior reporting date completed: 1.4sec


In [8]:
# get previous report date from extracted file names in the PGF hedge report folder
# https://www3.ntu.edu.sg/home/ehchua/programming/howto/Regexe.html

start_time = time.time()
print('Getting latest file ...')

fls = [int(re.match('\d{8}', str(k)).group()) for k in os.listdir(pth) if re.match('\d{8}', str(k)) != None]
fln = os.path.join(pth, str(max(fls)) + ' PGF Share Class Hedges.xlsx')
print(' ', fln)

print(f'Getting latest file completed: {timediff(start_time, time.time())}','\n')

Getting latest file ...
  P:\Investment Operations\GRC\Compliance\PGF UCITS Share Class Hedges\20250107 PGF Share Class Hedges.xlsx
Getting latest file completed: 0.0sec 



In [11]:
# prepare the free cover dataframe based on current and prior day derivative summary sheets

start_time = time.time()
print(f'Saving the free cover dataframe for {rDate.strftime("%a %d %b %Y")} vs {prrDate.strftime("%a %d %b %Y")}  ...')

# get fund long names from the 2AAX file
names   = pd.read_excel(pthFundCodes, sheet_name = 'Funds', usecols = ['Fund Code', 'Fund Name', 'Comment'])

# get latest derivative summary sheet as a dataframe
cols_dv = ['Fund Mandate', 'Cash Cover', 'UT?', '#', 'PIM Overdrafts']
a       = pd.read_excel(fr'{pthEXPORTS}' + fr'\Derv {rptDate}.xlsx', sheet_name = 'Summary', usecols = cols_dv)
a       = a.rename(columns = {'Cash Cover': 'Free Cover (% NAV)', 'PIM Overdrafts': 'Comment'})

# configure the column headings by adding two extra columns and ...
cols = ['Fund Name', 'Free Cover prior day (% NAV)', 'Trait']
for cols in cols: # create an empty summary dataframe with the given column headings
    # https://www.reddit.com/r/learnpython/comments/n1ee17/how_to_add_multiple_empty_columns_into_my_data/
    a[cols]=''

# ... reaaranging the column headings
heads  = ['Fund Mandate', 'Fund Name', 'Free Cover (% NAV)', 'Free Cover prior day (% NAV)', 'Comment', 'UT?', '#', 'Trait']
a      = a[heads]

# add prior day values to the dataframe
for row, name in enumerate(a['Fund Mandate']):
    a.iat[row, 1] = names[names['Fund Code'] == name].iat[0,1] # add fund long name to second column of the comparative summary
    a.iat[row, 7] = names[names['Fund Code'] == name].iat[0,2] # add comment to eighth column of the comparative summary
    try: # in the event that fund was added or removed from the derivative check list
        a.iat[row, 3] = pr_day[pr_day['Fund Mandate'] == name].iat[0,0] # add prior day value for that fund to fourth column of the comparative summary
    except:
        print(f" {name} wasn't there yesterday")

# sort the dataframe by least cover to most cover
a.sort_values('Free Cover (% NAV)', axis = 0, inplace = True)

# save the unsorted free cover dataframe as a file
rpt_day = datetime.datetime.strptime(rptDate,"%d%b%Y").strftime("%Y%m%d")
a.to_excel(pthFreeCover + fr'/{rpt_day} Free Cover.xlsx',index = False, sheet_name = 'Free Cover')

print(f'Saving the free cover dataframe for {rDate.strftime("%a %d %b %Y")} vs {prrDate.strftime("%a %d %b %Y")} \
completed: {timediff(start_time, time.time())}', '\n')

Saving the free cover dataframe for Mon 30 Dec 2024 vs Fri 27 Dec 2024  ...
Saving the free cover dataframe for Mon 30 Dec 2024 vs Fri 27 Dec 2024 completed: 2.7sec 



In [ ]:
# Prettify the free cover summary sheet and add hyperlinks

print(f'Prettifying and adding links to the summary sheet and then saving it ...')
start_time = time.time()

wbS    = xw.Book(r'P:\Investment Operations\GRC\Compliance\Daily\Free Cover.xlsm')
shtS   = wbS.sheets['Summary']                         # derivative cover summary sheet
#xl.DisplayAlerts = False                              # suppress Excel warning dialogues

# format the headings row of the Summary file
shtS['A1:E1'].api.WrapText = True
shtS['A1:E1'].font.bold    = True
shtS['A1:E1'].color        = (242, 242, 242) # light grey for the column headings

# add hyperlinks to each fund derivative calculation file
#for index, fund in enumerate(a_summarised['Fund Code']):
#    shtS['A' + str(index + 2)].add_hyperlink(pthEXPORTS + fr'\{fund} Derv Calc {rptDate}.xlsx',fund)

# set column widths
widths = {'A': 11, 'B': 50, 'C': 8, 'D': 8, 'E': 60}
for col in widths:
    shtS[f'{col}' + '1'].column_width = widths[col]

# add conditional formating for values that are negative or exceed 100% of NAV
for cell in shtS['C2:D2'].expand('down'):
    #if type(cell.value) in [float, int]:
    if cell.value < 0:
        cell.font.color = (255,   0,   0) # red is (255,0,0) in RGB or #FF0000 in Hex  

wbS.save()
wbS.close()
print(f'Prettifying and adding links to the summary sheet and then saving it completed: {timediff(start_time, time.time())}', '\n')

Prettifying and adding links to the summary sheet and then saving it ...


In [143]:
print('\n', f'Roundtrip time to complete free derivative cover file: {timediff(start_time_overlord, time.time())}', '\n')


 Roundtrip time to complete free derivative cover file: 16.7sec 



In [144]:
# Free cover template      P:\Investment Operations\GRC\Compliance\Daily\Free Cover.xlsm
# Exports folder           P:\Investment Operations\GRC\Compliance\Free Cover\
# derv cover calc template \\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\Daily\derv.xlsx